# Test Skill Vector Similarity

This notebook allows you to test the similarity between different skills/words using the exact same model (`BAAI/bge-small-en-v1.5`) and distance metric (Normalized L2 Distance) as the Vector Match API. You can use this to tune your Strong/Partial match thresholds.

In [1]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Load the model (this will use the cached version if already downloaded)
model = SentenceTransformer("BAAI/bge-small-en-v1.5")
print("Model loaded successfully!")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Model loaded successfully!


In [2]:
def calculate_similarity(skill1, skill2):
    # Embed and normalize to unit length (just like the API does)
    emb1 = model.encode([skill1], normalize_embeddings=True).astype(np.float32)
    emb2 = model.encode([skill2], normalize_embeddings=True).astype(np.float32)
    
    # Calculate L2 Distance
    l2_distance = np.linalg.norm(emb1 - emb2)
    
    # Calculate Cosine Similarity for reference (1 - (L2^2)/2)
    cosine_sim = 1 - (l2_distance**2) / 2
    
    print(f"\"{skill1}\" vs \"{skill2}\"")
    print(f"- L2 Distance:       {l2_distance:.4f} (Lower is closer)")
    print(f"- Cosine Similarity: {cosine_sim:.4f} (Higher is closer)")
    
    # These are the exact thresholds used in the API
    if l2_distance <= 0.55:
        print("- API Result:        ✅ STRONG MATCH (1.0 points)\n")
    elif l2_distance <= 0.80:
        print("- API Result:        ⚠️ PARTIAL MATCH (0.5 points)\n")
    else:
        print("- API Result:        ❌ NO MATCH (0 points)\n")


In [8]:
# Test your own word pairs here!

pairs_to_test = [
    ("Python", "Frontend Developme"),
    ("SQL Database", "Postgres Database"),
    ("Python", "Python Programming"),
    ("Machine Learning", "Deep Learning"),
    ("Frontend Development", "UI/UX Design"),
    ("React", "React JS")

]

for skill_a, skill_b in pairs_to_test:
    calculate_similarity(skill_a, skill_b)


"Python" vs "Frontend Developme"
- L2 Distance:       0.9543 (Lower is closer)
- Cosine Similarity: 0.5446 (Higher is closer)
- API Result:        ❌ NO MATCH (0 points)

"SQL Database" vs "Postgres Database"
- L2 Distance:       0.6001 (Lower is closer)
- Cosine Similarity: 0.8199 (Higher is closer)
- API Result:        ⚠️ PARTIAL MATCH (0.5 points)

"Python" vs "Python Programming"
- L2 Distance:       0.4492 (Lower is closer)
- Cosine Similarity: 0.8991 (Higher is closer)
- API Result:        ✅ STRONG MATCH (1.0 points)

"Machine Learning" vs "Deep Learning"
- L2 Distance:       0.6143 (Lower is closer)
- Cosine Similarity: 0.8113 (Higher is closer)
- API Result:        ⚠️ PARTIAL MATCH (0.5 points)

"Frontend Development" vs "UI/UX Design"
- L2 Distance:       0.7198 (Lower is closer)
- Cosine Similarity: 0.7409 (Higher is closer)
- API Result:        ⚠️ PARTIAL MATCH (0.5 points)

"React" vs "React JS"
- L2 Distance:       0.4422 (Lower is closer)
- Cosine Similarity: 0.9022 (Highe